# Search Strategy Refinement — Method Comparison

*Author: Regina Chua*

> This notebook tests three complementary methods for refining the search strategy and pre-ranking
> the corpus against a small set of manually confirmed seed papers. The goal is to compare which
> method surfaces the most relevant articles and what new vocabulary it suggests — feeding back into
> `search_strategy.py` before the full multi-database run.

**Methods tested (all recommended in `planningnotes.md`):**

| Method | What it does | Best for |
|---|---|---|
| **BM25** | Term-frequency ranking with length normalisation | Keyword recall, interpretable scores |
| **KeyBERT / YAKE** | Keyphrase extraction from seed abstracts | Surfacing new vocabulary for query expansion |
| **SPECTER embeddings** | Semantic similarity in dense vector space | Finding papers that use different terminology |

**Workflow:** populate `seed_papers.csv` → run top-to-bottom → inspect rankings and candidate
keyphrases → update `search_strategy.py` with any new terms.

---

### Seed set guidance

> Your planning notes flag 7 papers as "a thin prior." For reliable ranking:
>
> | Size | What it enables |
> |---|---|
> | **≥ 15** | Minimum for stable BM25 and embedding centroids |
> | **20–30** | Recommended — covers enough sub-theme diversity |
> | **≥ 50** | Needed for the calibration step (precision/recall tuning) |
>
> Papers should be ones you have **manually confirmed as relevant** to the review (i.e. you would
> include them at full-text screening). They do not need to have come from PubMed — any source is fine.
> Each paper needs at minimum a **title and abstract**; DOI and PubMed ID are helpful but optional.

## 1. Environment Setup

> Install the three extra libraries if needed, then import everything. `sentence-transformers`
> downloads the SPECTER model on first use (~400 MB); subsequent runs load it from cache.

In [ ]:
%pip install rank-bm25 sentence-transformers keybert yake --quiet

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 100)

SEED_PATH   = Path("seed_papers.csv")
CORPUS_PATH = Path("pubmed_results_cleaned_2026.csv")
TOP_N       = 20   # how many top-ranked corpus papers to show per method

print("Environment ready.")

## 2. Load Seed Papers & Corpus

> The seed set is loaded from `seed_papers.csv` (in the project root). Each row must have at
> least a `title` and `abstract` — the richer the abstract text, the better all three methods
> will perform. The corpus is the cleaned PubMed results; swap in SCOPUS/EMBASE CSVs once those
> are collected.

In [ ]:
def _combine_text(row):
    """Join title, abstract, and keywords into one string for indexing."""
    parts = [
        str(row.get("title", "") or ""),
        str(row.get("abstract", "") or ""),
        str(row.get("keywords", "") or ""),
    ]
    return " ".join(p for p in parts if p).strip()


def _tokenize(text):
    """Lowercase, strip punctuation, split on whitespace."""
    return re.sub(r"[^a-z0-9\s]", " ", text.lower()).split()


# --- Seed papers ---
if not SEED_PATH.exists():
    raise FileNotFoundError(
        f"'{SEED_PATH}' not found. Add your confirmed-relevant papers to that file "
        "(title + abstract required) and re-run this cell."
    )

df_seed = pd.read_csv(SEED_PATH)
# Drop the placeholder row if it hasn't been replaced yet
df_seed = df_seed[~df_seed["title"].astype(str).str.startswith("REPLACE")].reset_index(drop=True)

if len(df_seed) == 0:
    raise ValueError(
        "seed_papers.csv contains no real entries yet. Fill in at least one confirmed-relevant "
        "paper (title + abstract) and re-run."
    )

df_seed["_text"] = df_seed.apply(_combine_text, axis=1)
seed_texts = df_seed["_text"].tolist()
print(f"Seed papers loaded: {len(df_seed)}")
if len(df_seed) < 15:
    print(f"  ⚠  {len(df_seed)} papers is below the recommended minimum of 15. "
          "Rankings will be less reliable — add more confirmed-relevant papers.")
display(df_seed[[c for c in ["title","doi","pubmed_id"] if c in df_seed.columns]].head())

# --- Corpus ---
if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"'{CORPUS_PATH}' not found. Run pubmed.ipynb first to generate it."
    )

df_corpus = pd.read_csv(CORPUS_PATH)
df_corpus["_text"] = df_corpus.apply(_combine_text, axis=1)
# Drop rows with no usable text
df_corpus = df_corpus[df_corpus["_text"].str.strip().astype(bool)].reset_index(drop=True)
print(f"\nCorpus loaded: {len(df_corpus)} documents from '{CORPUS_PATH.name}'")

## 3. Method 1 — BM25

> BM25 is a direct upgrade over TF-IDF (already in `pubmed.ipynb`): it adds document-length
> normalisation and term saturation so high-frequency terms don't dominate. I build a query from
> the concatenated seed texts and score every corpus document. The output is a ranked list of the
> most BM25-relevant articles — useful both as a sanity check (known-relevant papers from the
> PubMed run should rank high) and as a complement to the embedding method below.

In [ ]:
from rank_bm25 import BM25Okapi

# Build the BM25 index over the corpus
corpus_tokens = [_tokenize(t) for t in df_corpus["_text"]]
bm25 = BM25Okapi(corpus_tokens)

# Query = all seed text concatenated
seed_query_tokens = _tokenize(" ".join(seed_texts))
bm25_scores = bm25.get_scores(seed_query_tokens)

df_bm25 = df_corpus.copy()
df_bm25["bm25_score"] = bm25_scores
df_bm25 = df_bm25.sort_values("bm25_score", ascending=False).reset_index(drop=True)

print(f"Top {TOP_N} BM25-ranked corpus documents:")
show_cols = [c for c in ["title", "bm25_score", "doi", "publication_date"] if c in df_bm25.columns]
display(df_bm25[show_cols].head(TOP_N).style.format({"bm25_score": "{:.2f}"}))

## 4. Method 2 — Keyphrase Extraction (KeyBERT + YAKE)

> Both methods extract keyphrases from the seed abstracts, but use different signals. **YAKE** uses
> statistical co-occurrence (fast, no model download). **KeyBERT** uses contextual embeddings (richer,
> slower). The union of their output is a candidate list of terms to add to `search_strategy.py`
> — specifically to `ALTERNATE_TERMS` or `INCLUSION_CRITERIA`. I mark any term that already exists
> in the strategy so it's easy to spot the genuinely new ones.

In [ ]:
import yake
from keybert import KeyBERT

from search_strategy import INCLUSION_CRITERIA, ALTERNATE_TERMS

# All terms already in the strategy (for deduplication display)
existing_terms = set()
for group in list(INCLUSION_CRITERIA.values()) + list(ALTERNATE_TERMS.values()):
    for t in group:
        existing_terms.add(t.lower().replace("*", ""))

seed_corpus_text = " ".join(seed_texts)

# --- YAKE ---
yake_extractor = yake.KeywordExtractor(
    lan="en", n=3, dedupLim=0.7, top=30, features=None
)
yake_kws = yake_extractor.extract_keywords(seed_corpus_text)
# YAKE scores are inverted (lower = more important)
yake_df = pd.DataFrame(yake_kws, columns=["keyphrase", "yake_score"]).sort_values("yake_score")
yake_df["already_in_strategy"] = yake_df["keyphrase"].str.lower().isin(existing_terms)

print("--- YAKE keyphrases (lower score = more relevant) ---")
display(yake_df.head(20).style.format({"yake_score": "{:.4f}"}))

# --- KeyBERT ---
# Uses a lightweight all-MiniLM model by default (fast). Swap for 'allenai-specter'
# if you want scientific-domain embeddings (requires the SPECTER model to be downloaded first).
print("\nLoading KeyBERT model (this may take a moment on first run)...")
kw_model = KeyBERT()
keybert_kws = kw_model.extract_keywords(
    seed_corpus_text,
    keyphrase_ngram_range=(1, 3),
    stop_words="english",
    top_n=30,
    diversity=0.6,   # MMR diversity — avoids near-duplicate keyphrases
)
keybert_df = pd.DataFrame(keybert_kws, columns=["keyphrase", "keybert_score"]).sort_values(
    "keybert_score", ascending=False
)
keybert_df["already_in_strategy"] = keybert_df["keyphrase"].str.lower().isin(existing_terms)

print("\n--- KeyBERT keyphrases (higher score = more relevant) ---")
display(keybert_df.head(20).style.format({"keybert_score": "{:.3f}"}))

# --- Union of new terms ---
new_yake = set(yake_df[~yake_df["already_in_strategy"]]["keyphrase"].str.lower())
new_keybert = set(keybert_df[~keybert_df["already_in_strategy"]]["keyphrase"].str.lower())
new_terms_union = sorted(new_yake | new_keybert)
print(f"\n{len(new_terms_union)} candidate new terms (not already in search_strategy.py):")
for t in new_terms_union:
    print(f"  {t}")

## 5. Method 3 — SPECTER Semantic Similarity

> SPECTER is a transformer model trained specifically for scientific document similarity. I embed
> each seed paper and each corpus document, then score each corpus document by its **cosine
> similarity to the seed centroid** (the average of the seed embeddings). This catches papers that
> are conceptually similar to the seed set but use different terminology — the gap that keyword
> methods leave open.
>
> First run downloads the SPECTER model (~400 MB); subsequent runs use the cached version.
> If the download is too slow, replace `'allenai-specter'` with `'all-MiniLM-L6-v2'` for a
> faster ~80 MB model (slightly less domain-specific).

In [ ]:
from sentence_transformers import SentenceTransformer, util

MODEL_NAME = "allenai-specter"   # swap for "all-MiniLM-L6-v2" if you want a faster, smaller model

print(f"Loading {MODEL_NAME} (downloads ~400 MB on first run, then cached) ...")
model = SentenceTransformer(MODEL_NAME)
print("Model loaded.")

# Embed seed papers
print("Embedding seed papers ...")
seed_embeddings = model.encode(seed_texts, show_progress_bar=True, convert_to_tensor=True)
seed_centroid = seed_embeddings.mean(dim=0)  # single representative vector

# Embed corpus
print("Embedding corpus (may take a few minutes) ...")
corpus_embeddings = model.encode(
    df_corpus["_text"].tolist(), show_progress_bar=True, convert_to_tensor=True
)

# Score each corpus document against the seed centroid
sim_scores = util.cos_sim(seed_centroid.unsqueeze(0), corpus_embeddings)[0].cpu().numpy()

df_specter = df_corpus.copy()
df_specter["specter_sim"] = sim_scores
df_specter = df_specter.sort_values("specter_sim", ascending=False).reset_index(drop=True)

print(f"\nTop {TOP_N} SPECTER-ranked corpus documents:")
show_cols = [c for c in ["title", "specter_sim", "doi", "publication_date"] if c in df_specter.columns]
display(df_specter[show_cols].head(TOP_N).style.format({"specter_sim": "{:.3f}"}))

## 6. Compare Rankings

> Merges the BM25 and SPECTER rankings into one table so it's easy to spot:
> - Papers that rank high on **both** — very likely relevant.
> - Papers high on SPECTER but low on BM25 — semantically similar but may use different vocabulary;
>   their titles/abstracts are good candidates for new search terms.
> - Papers high on BM25 but low on SPECTER — keyword-rich but may be thematically tangential;
>   worth a quick scan to check whether they are noise or whether the seed set is missing a sub-theme.

In [ ]:
# Normalise both scores to [0, 1] for a fair side-by-side comparison
def _norm(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else s * 0


bm25_rank = df_bm25[["title", "doi", "bm25_score"]].copy()
bm25_rank["bm25_norm"] = _norm(bm25_rank["bm25_score"])
bm25_rank["bm25_rank"] = bm25_rank["bm25_score"].rank(ascending=False).astype(int)

specter_rank = df_specter[["title", "doi", "specter_sim"]].copy()
specter_rank["specter_norm"] = _norm(specter_rank["specter_sim"])
specter_rank["specter_rank"] = specter_rank["specter_sim"].rank(ascending=False).astype(int)

merged = bm25_rank.merge(specter_rank[["doi", "specter_norm", "specter_rank"]], on="doi", how="inner")
merged["combined_norm"] = (merged["bm25_norm"] + merged["specter_norm"]) / 2
merged = merged.sort_values("combined_norm", ascending=False).reset_index(drop=True)

print(f"Top {TOP_N} papers by combined BM25 + SPECTER score:")
display(
    merged[["title", "bm25_rank", "specter_rank", "bm25_norm", "specter_norm", "combined_norm"]]
    .head(TOP_N)
    .style.format({
        "bm25_norm":     "{:.3f}",
        "specter_norm":  "{:.3f}",
        "combined_norm": "{:.3f}",
    })
    .background_gradient(subset=["combined_norm"], cmap="YlGn")
)

# Flag interesting divergences
divergent = merged[abs(merged["bm25_rank"] - merged["specter_rank"]) > 50].head(10)
if not divergent.empty:
    print(f"\nPapers with large rank divergence (>50 positions) — worth investigating:")
    display(divergent[["title", "bm25_rank", "specter_rank"]].reset_index(drop=True))

## 7. Export Ranked Results

> Save all three score columns to a CSV so the rankings can be reviewed outside the notebook and
> used as a reference when updating `search_strategy.py`.

In [ ]:
from pathlib import Path

# Attach scores back to the full corpus metadata
full_ranked = df_corpus.drop(columns=["_text"]).merge(
    merged[["doi", "bm25_rank", "specter_rank", "bm25_norm", "specter_norm", "combined_norm"]],
    on="doi",
    how="left",
)
full_ranked = full_ranked.sort_values("combined_norm", ascending=False).reset_index(drop=True)

output_path = Path("query_refinement_ranked.csv")
full_ranked.to_csv(output_path, index=False)
print(f"Exported {len(full_ranked)} ranked records to {output_path.resolve()}")
print()
print("Next steps:")
print("  1. Review the candidate new terms from Section 4 and add useful ones to")
print("     ALTERNATE_TERMS (or INCLUSION_CRITERIA) in search_strategy.py.")
print("  2. Skim the top-ranked papers from Section 6 that are NOT already in your seed set")
print("     — any you would include become additional seed papers.")
print("  3. If enough papers diverge between BM25 and SPECTER, consider adding a sub-theme")
print("     to the seed set to cover that vocabulary gap.")
print("  4. Once you have ≥50 labelled papers (include + confirmed exclude), run the")
print("     calibration step: score a held-out set and measure precision/recall to tune")
print("     the similarity thresholds before running LLM screening in prescreen.ipynb.")